# 01 — Extracció del cas d'ús: Ruta DR0051 Vic

**Hackathon Interhack BCN 2026 — Repte Damm Smart Truck**

Aquest notebook fa 4 coses:
1. Carrega les dades reals del transport `11443257` (5/2/2026, ruta DR0051 zona Vic, 24 clients).
2. Calcula els palets equivalents per cada client (a partir del mestre de materials ZM040).
3. Afegeix les finestres horàries dels horaris d'entrega.
4. Genera el CSV `data/processed/cas_us_clients.csv` que farà servir tot l'equip.

La geocodificació (lat/lon) es fa al notebook `02_geocodificacio.ipynb` perquè és lenta i requereix internet.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutes (relatives a /notebooks/)
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True, parents=True)

# Cas d'ús: ruta de Vic del 5/2/2026
TRANSPORT_ID = 11443257

print('Setup OK')

Setup OK


## 1. Carregar el transport del cas d'ús

In [2]:
df = pd.read_excel(RAW / 'Hackaton.xlsx', sheet_name='Detalle entrega')
print(f'Dataset complet: {len(df):,} línies, {df["Transporte"].nunique()} transports')

ruta = df[df['Transporte'] == TRANSPORT_ID].copy()
print(f'\nTransport {TRANSPORT_ID}:')
print(f'  Data: {ruta["FECHA"].iloc[0]}')
print(f'  Ruta: {ruta["Ruta"].iloc[0]}')
print(f'  Repartidor: {ruta["Repartidor"].iloc[0]}')
print(f'  Clients diferents: {ruta["Destinatario mcía..1"].nunique()}')
print(f'  Línies de comanda: {len(ruta)}')

Dataset complet: 82,849 línies, 889 transports

Transport 11443257:
  Data: 05/02/2026
  Ruta: DR0051
  Repartidor: 855190
  Clients diferents: 24
  Línies de comanda: 196


In [3]:
# Resum per client
clients = (ruta
    .groupby('Destinatario mcía..1')
    .agg(
        nom=('Nombre 1', 'first'),
        carrer=('Calle', 'first'),
        cp=('CP', 'first'),
        poblacio=('Población', 'first'),
        n_linies=('Material', 'count')
    )
    .reset_index()
    .rename(columns={'Destinatario mcía..1': 'client_id'})
)
clients

,client_id,nom,carrer,cp,poblacio,n_linies
0,136675,MGA GP VIC,"CTRA. DE SANT HIPÒLIT, 53 (NAU 4)",8500,VIC,2
1,9100058446,GUSTUM AREAS 1 (LLEIDA-GIRONA),"CARRETERA C-25 EIX TRANSVE KM 174,6",8503,GURB,7
2,9100058476,GUSTUM AREAS 2 (GIRONA-LLEIDA),"CARRETERA C-25 EIX TRANSVE KM 174,6",8503,GURB,7
3,9100058727,SNACK (VIC),Plaça Major 28,8500,VIC,13
4,9100058850,BAR LA PISTA,CALLE SOLEDAT 4,8500,VIC,6
5,9100134828,EL CALIU RESTAURANT (VIC),CALLE RIERA 13,8500,VIC,18
6,9100158925,VINS I BEGUDES REMEI,CALLE VIRREI AVILES 28,8500,VIC,4
7,9100324575,BAR SOL NOU,PASEO GENERALITAT 8,8500,VIC,12
8,9100374304,BAR PRIEGO,CALLE ENRIC PRAT DE LA RIBA 49,8500,VIC,12
9,9100374429,LA TAVERNA D'EN NORTON,PLAZA PES 6,8500,VIC,6


## 2. Calcular palets equivalents

Fem servir el mestre de materials `ZM040.xlsx`. Per cada material, agafem la fila amb `UMA=PAL` (palet) que ens diu quantes caixes hi caben.

**Lògica simple**:
- Si la unitat de venda és `CAJ` (caixa) → palets = quantitat / caixes_per_palet
- Si és `BRL` (barril), `BOT`, `UN`, etc. → assumim 0.1 palets per unitat (placeholder, refinarem si dona temps)

In [4]:
zm040 = pd.read_excel(RAW / 'ZM040.XLSX')
print(f'Mestre materials: {len(zm040):,} files, {zm040["Material"].nunique():,} materials únics')

# Agafem només les files amb UMA=PAL i el camp Contador (caixes per palet)
palets_info = (zm040[zm040['UMA'] == 'PAL']
    .groupby('Material', as_index=False)
    .agg(
        caixes_per_palet=('Contador', 'first'),
        volum_palet_l=('Volumen', 'first'),
        pes_palet_kg=('Peso bruto', 'first')
    )
)
print(f'Materials amb info de palet: {len(palets_info):,}')
palets_info.head()

Mestre materials: 48,457 files, 7,478 materials únics
Materials amb info de palet: 4,027


,Material,caixes_per_palet,volum_palet_l,pes_palet_kg
0,0AG0001,84.0,0.0,798.0
1,0AG0002,60.0,0.0,864.0
2,0AG0003,32.0,0.0,500.0
3,0AG0004,63.0,0.0,819.0
4,0AG0005,40.0,0.0,0.0


In [5]:
# Unim la ruta amb la info de palets
ruta_pal = ruta.merge(palets_info, on='Material', how='left')

# Quants materials de la ruta no tenen info de palet?
sense_info = ruta_pal['caixes_per_palet'].isna().sum()
print(f'Línies amb info de palet: {len(ruta_pal) - sense_info} / {len(ruta_pal)}')
print(f'Línies SENSE info: {sense_info}')

if sense_info > 0:
    print('\nMaterials sense info de palet:')
    print(ruta_pal[ruta_pal['caixes_per_palet'].isna()][['Material', 'Denominación', 'Un.medida venta']].drop_duplicates().head(10))

Línies amb info de palet: 112 / 196
Línies SENSE info: 84

Materials sense info de palet:
    Material                              Denominación Un.medida venta
2       CJ13  CAJA DAMM+BOT.1/3RET VACIO EN P.PLAS.A13             CAJ
8      CJ11V                     CAJAS COMPLETAS 1/1 V             CAJ
10     CJ12V                     CAJAS COMPLETAS 1/2 V             CAJ
16    BRL30V                BARRIL INOX 30L. EURONORMA             BRL
18    BRL20V             BARRIL INOX 20L. TANQUETA A16             BRL
25   0AM1012                  CHOVI MAYONESA CLAS. 5KG              UN
38   0VE2628          TORELLO CAVA BRUT NATURE 75CL 1U              UN
42  3ENV0236             C.C. AGUA 1/3 VICHY-FONT D'OR             CAJ
44  3ENV0029                       C.C. CACAOLAT 30 U.             CAJ
48      CJ15  CAJA DAMM+BOT.1/5RET VACIO EN P.PLAS.A13             CAJ


In [6]:
# Calcular palets equivalents per cada línia
def palets_eq(row):
    qty = row['Cantidad entrega']
    unitat = row['Un.medida venta']
    cpp = row['caixes_per_palet']
    
    if pd.isna(cpp) or cpp <= 0:
        # No tenim info → placeholder
        return 0.1
    
    if unitat == 'CAJ':
        return qty / cpp
    elif unitat == 'BRL':  # Barrils (cervesa pressió) - es transporten apilats
        return qty / 36  # ~36 barrils per palet (estimació conservadora)
    elif unitat in ('BOT', 'UN'):
        return qty / (cpp * 6) if cpp > 0 else 0.05
    else:
        return 0.1

ruta_pal['palets_equiv'] = ruta_pal.apply(palets_eq, axis=1)

# Total palets per client
palets_per_client = (ruta_pal
    .groupby('Destinatario mcía..1')['palets_equiv']
    .sum()
    .round(2)
    .reset_index()
    .rename(columns={'Destinatario mcía..1': 'client_id', 'palets_equiv': 'palets'})
)

print(f'Palets totals de la ruta: {palets_per_client["palets"].sum():.1f}')
print(f'Capacitat camió 6 palets: {6}')
print(f'Capacitat camió 8 palets: {8}')
print()
palets_per_client.sort_values('palets', ascending=False)

Palets totals de la ruta: 12.7
Capacitat camió 6 palets: 6
Capacitat camió 8 palets: 8



,client_id,palets
5,9100134828,1.55
3,9100058727,1.45
12,9100517701,1.02
13,9100564797,0.91
7,9100324575,0.85
8,9100374304,0.85
4,9100058850,0.61
20,9100745910,0.60
11,9100397795,0.47
1,9100058446,0.45


> **NOTA**: si surt > 6 palets, és l'esperat. La ruta original era amb un camió més gran o amb un mètode d'estiba que comparteix palets entre clients. Per al MVP, treballarem amb el supòsit que cap client supera 1 palet (els palets es comparteixen). Si voleu refinar-ho després, es pot.

## 3. Afegir finestres horàries

In [ ]:
horarios = pd.read_excel("raw/Horarios_Entrega.XLSX")
print(f'Horaris: {len(horarios)} files, {horarios["Deudor"].nunique()} clients amb finestra')
horarios.head(3)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# La data del cas d'ús és 5/2/2026 (dijous = dia 4)
# Filtrem horaris pel dijous, torn 1 (matí)
DIA_SETMANA = 4  # dijous
TORN = 1

horaris_dia = horarios[(horarios['Día semana'] == DIA_SETMANA) & (horarios['Turno'] == TORN)].copy()
print(f'Horaris dijous matí: {len(horaris_dia)} clients')

# El camp Deudor són els últims 6 dígits del client_id (91xxxxxxxx)
# Anem a unir-ho amb els clients del cas d'ús
# Primer mirem si els 24 clients de la ruta tenen horari

# El Deudor sembla un id intern de SAP. Comprovem si coincideix amb els últims dígits del client
horaris_dia['client_id_match'] = '91' + horaris_dia['Deudor'].astype(str).str.zfill(8)
horaris_dia['client_id_match'] = horaris_dia['client_id_match'].astype(int)

match = horaris_dia[horaris_dia['client_id_match'].isin(clients['client_id'])]
print(f'Clients del cas d\'ús amb finestra horària explícita: {len(match)} / {len(clients)}')

In [ ]:
# Per als que no tenen horari explícit, assumim finestra àmplia 08:00-18:00
# Per als que sí en tenen, agafem la franja

horaris_select = match[['client_id_match', 'Horario inicia a', 'Horario termina a']].rename(
    columns={'client_id_match': 'client_id', 
             'Horario inicia a': 'finestra_inici',
             'Horario termina a': 'finestra_fi'}
)
horaris_select = horaris_select.drop_duplicates('client_id')
horaris_select.head()

## 4. Generar el CSV final

In [ ]:
# Unim tot
final = clients.merge(palets_per_client, on='client_id', how='left')
final = final.merge(horaris_select, on='client_id', how='left')

# Default per als que no tenen finestra: 08:00-18:00
from datetime import time
final['finestra_inici'] = final['finestra_inici'].fillna(time(8, 0))
final['finestra_fi'] = final['finestra_fi'].fillna(time(18, 0))

# Adreça completa per a la geocodificació posterior
final['adreca_completa'] = (
    final['carrer'].astype(str) + ', ' +
    final['cp'].astype(str).str.replace('.0', '', regex=False).str.zfill(5) + ' ' +
    final['poblacio'].astype(str) + ', España'
)

# Reordenem columnes
final = final[['client_id', 'nom', 'carrer', 'cp', 'poblacio', 'adreca_completa',
               'palets', 'finestra_inici', 'finestra_fi', 'n_linies']]

final

In [ ]:
# Guardar
OUT = PROCESSED / 'cas_us_clients.csv'
final.to_csv(OUT, index=False)
print(f'✓ Guardat a {OUT}')
print(f'  {len(final)} clients')
print(f'  {final["palets"].sum():.1f} palets totals')

# També guardem el detall complet de la ruta (totes les línies amb material) per si ho necessitem després
ruta_pal.to_csv(PROCESSED / 'cas_us_linies.csv', index=False)
print(f'✓ Detall complet a {PROCESSED / "cas_us_linies.csv"}')

## Següent pas

Anar al notebook **`02_geocodificacio.ipynb`** per obtenir lat/lon de cada client a partir de l'adreça.